In [1]:
import requests
import pandas as pd
import time
import json
from pathlib import Path

In [2]:
# =====================================
# CONFIG
# =====================================

API_KEYS = [
    "4d2fda2a8e97ad11381e02ab10eb2e90dac641dc7d48e5bc069063c2bb53f57a", 
    "926fe3ea394a551008625a0b8985b5782bc7a5523c12b07b822a675440539a15",
    "1e14555c5f2cb18790d33179b0f3a43e081d7e8a9ac02608f50dc65675c55357", 
    "b9bca9489723e7fcc8fcf454d465c50dd8f9562941f559c8cfb259947f608bfd", 
    "86576cf07cc6131f4173075d494d95ccc83f79fe0f26cd9979cf8c2bf5fade68", 
    "f2035999fed92a2ff503e83ea4a2fd50b4dd1a9023ea8c30a71ba56addde1dba" 
]

current_key_idx = 0

BASE_URL = "https://api.ceda.ashoka.edu.in/v1/agmarknet"

STATE_ID = 9          # Uttar Pradesh
COMMODITY_ID = 1      # Wheat

OUTDIR = Path("UP_WHEAT_2006_2025")
OUTDIR.mkdir(exist_ok=True)

MARKET_MAP_FILE = "district_market_map.json"

current_key_idx = 0
successful_requests = 0
key_stats = {
    i: {
        "success_requests": 0,
        "rate_limits": 0
    }
    for i in range(len(API_KEYS))
}

In [3]:
# =====================================
# AUTH HEADER
# =====================================

def get_headers():

    return {
        "Authorization":
            f"Bearer {API_KEYS[current_key_idx]}",
        "Content-Type":
            "application/json"
    }

# =====================================
# KEY ROTATION
# =====================================

def rotate_key():

    global current_key_idx

    key_stats[current_key_idx]["rate_limits"] += 1

    print("\n" + "=" * 60)

    print(
        f"KEY {current_key_idx + 1} EXHAUSTED"
    )

    print(
        f"Successful requests: "
        f"{key_stats[current_key_idx]['success_requests']}"
    )

    print("=" * 60)

    # Last key exhausted

    if current_key_idx + 1 >= len(API_KEYS):

        print(
            "\nALL API KEYS EXHAUSTED"
        )

        return False

    current_key_idx += 1

    print(
        f"\nSwitched to API key "
        f"{current_key_idx + 1}"
    )

    return True

# =====================================
# POST WITH RETRY
# =====================================

def post_with_retry(
    endpoint,
    payload,
    max_retries=3
):

    url = f"{BASE_URL}/{endpoint}"

    for attempt in range(max_retries):

        try:

            r = requests.post(
                url,
                headers=get_headers(),
                json=payload,
                timeout=120
            )

            # ---------------------------------
            # PARSE RESPONSE
            # ---------------------------------

            try:

                data = r.json()

            except Exception:

                print(
                    f"Invalid JSON "
                    f"(attempt {attempt + 1})"
                )

                time.sleep(5)

                continue

            # ---------------------------------
            # RATE LIMIT
            # ---------------------------------

            if (
                r.status_code == 429
                or (
                    isinstance(data, dict)
                    and
                    "Too many requests"
                    in str(data)
                )
            ):

                ok = rotate_key()

                if not ok:

                    # ALL KEYS EXHAUSTED

                    return None

                continue

            # ---------------------------------
            # HTTP ERRORS
            # ---------------------------------

            if r.status_code != 200:

                print(
                    f"HTTP Error "
                    f"{r.status_code}"
                )

                time.sleep(5)

                continue

            # ---------------------------------
            # SUCCESS
            # ---------------------------------

            key_stats[
                current_key_idx
            ][
                "success_requests"
            ] += 1

            count = key_stats[
                current_key_idx
            ][
                "success_requests"
            ]

            if count % 5 == 0:

                print(
                    "\n"
                    + "=" * 50
                )

                print(
                    f"KEY "
                    f"{current_key_idx + 1}"
                )

                print(
                    f"Successful requests: "
                    f"{count}"
                )

                print(
                    "=" * 50
                )

            return data

        except requests.exceptions.Timeout:

            print(
                f"Timeout "
                f"(attempt {attempt + 1})"
            )

        except requests.exceptions.ConnectionError:

            print(
                f"Connection Error "
                f"(attempt {attempt + 1})"
            )

        except Exception as e:

            print(
                f"Retry {attempt + 1}: "
                f"{e}"
            )

        time.sleep(10)

    # ---------------------------------
    # REQUEST FAILED AFTER RETRIES
    # ---------------------------------

    return {
        "status": "request_failed"
    }

In [14]:
# =====================================
# GET DISTRICTS
# =====================================

print(
    "Downloading geography data..."
)

geo = requests.get(
    f"{BASE_URL}/geographies",
    headers=get_headers()
).json()

districts = pd.DataFrame(
    geo["output"]["data"]
)

up_districts = districts[
    districts["census_state_id"]
    == STATE_ID
].copy()

print(
    f"Found "
    f"{len(up_districts)} districts"
)

Found 71 districts


In [15]:
# =====================================
# GET MARKETS
# =====================================

def get_markets(district_id):

    payload = {

        "commodity_id":
            COMMODITY_ID,

        "state_id":
            STATE_ID,

        "district_id":
            int(district_id),

        "indicator":
            "price"
    }

    return post_with_retry(
        "markets",
        payload
    )

In [16]:
# =====================================
# LOAD EXISTING MARKET MAP
# =====================================

if Path(
    MARKET_MAP_FILE
).exists():

    with open(
        MARKET_MAP_FILE,
        "r"
    ) as f:

        district_market_map = (
            json.load(f)
        )

    district_market_map = {

        int(k): v

        for k, v in
        district_market_map.items()

    }

    print(
        f"Loaded "
        f"{len(district_market_map)} "
        f"districts"
    )

else:

    district_market_map = {}

# =====================================
# DISCOVER MARKETS
# =====================================

for _, row in up_districts.iterrows():

    district_id = int(
        row["census_district_id"]
    )

    district_name = row[
        "census_district_name"
    ]

    if district_id in district_market_map:

        print(
            f"Skipping "
            f"{district_name}"
        )

        continue

    print(
        f"\nMarkets: "
        f"{district_name}"
    )

    try:

        market_response = (
            get_markets(
                district_id
            )
        )

        if market_response is None:
            continue

        if (
            "output"
            not in market_response
        ):
            print(
                market_response
            )
            continue

        if (
            market_response[
                "output"
            ].get("type")
            != "success"
        ):
            print(
                market_response
            )
            continue

        markets = (
            market_response[
                "output"
            ]["data"]
        )

        district_market_map[
            district_id
        ] = {

            "name":
                district_name,

            "markets": [

                {
                    "market_id":
                        m["market_id"],

                    "market_name":
                        m["market_name"]
                }

                for m in markets

            ]
        }

        with open(
            MARKET_MAP_FILE,
            "w"
        ) as f:

            json.dump(
                district_market_map,
                f,
                indent=2
            )

        print(
            f"Found "
            f"{len(markets)} markets"
        )

        time.sleep(2)

    except Exception as e:

        print(
            district_name,
            e
        )

print(
    "\n================================="
)

print(
    "Districts with markets:",
    len(
        district_market_map
    )
)

print(
    "================================="
)

Loaded 3 districts
Skipping Saharanpur
Skipping Muzaffarnagar
Skipping Bijnor

Markets: Moradabad
Found 4 markets

Markets: Rampur
Found 5 markets

Markets: Jyotiba Phule Nagar
Found 3 markets

Markets: Meerut
Found 4 markets

Markets: Baghpat
Found 2 markets

Markets: Ghaziabad
Found 4 markets

Markets: Gautam Buddha Nagar
Found 3 markets

Markets: Bulandshahr
Found 9 markets

Markets: Aligarh
Found 4 markets

Markets: Mahamaya Nagar
Found 3 markets

Markets: Mathura
Found 2 markets

Markets: Agra
Found 8 markets

Markets: Firozabad
Found 4 markets

Markets: Mainpuri
Found 3 markets

Markets: Budaun
Found 8 markets

Markets: Bareilly
Found 4 markets

Markets: Pilibhit
Found 4 markets

Markets: Shahjahanpur
Found 6 markets

Markets: Kheri
Found 6 markets

Markets: Sitapur
Found 7 markets

Markets: Hardoi
Found 5 markets

Markets: Unnao
Found 3 markets

Markets: Lucknow
Found 2 markets

Markets: Rae Bareli
Found 5 markets

Markets: Farrukhabad
Found 5 markets

Markets: Kannauj
Found 2 m

In [18]:
import json

with open("district_market_map.json") as f:
    district_market_map = json.load(f)

print(len(district_market_map))

71


In [19]:
import shutil

shutil.copy(
    "district_market_map.json",
    "district_market_map_backup.json"
)

'district_market_map_backup.json'

In [20]:
import json

with open("district_market_map.json") as f:
    district_market_map = json.load(f)

sample_district = next(iter(district_market_map))

sample_market = district_market_map[
    sample_district
]["markets"][0]["market_id"]

payload = {
    "commodity_id": COMMODITY_ID,
    "state_id": STATE_ID,
    "district_id": int(sample_district),
    "market_id": sample_market,
    "from_date": "2024-01-01",
    "to_date": "2024-12-31"
}

resp = post_with_retry(
    "prices",
    payload
)

print(resp)

Rate limit on key 1

Switched to API key 2
{'output': {'type': 'success', 'message': 'No data exists', 'data': []}}


In [21]:
import json

with open("district_market_map.json") as f:
    district_market_map = json.load(f)

sample_district = next(iter(district_market_map))

print("District:", sample_district)
print(district_market_map[sample_district])

District: 132
{'name': 'Saharanpur', 'markets': [{'market_id': 2544, 'market_name': 'Nakud'}, {'market_id': 1960, 'market_name': 'Rampurmaniharan'}, {'market_id': 1957, 'market_name': 'Deoband'}, {'market_id': 2543, 'market_name': 'Chutmalpur'}, {'market_id': 2546, 'market_name': 'Sultanpurchilkana'}, {'market_id': 2545, 'market_name': 'Nanuta'}, {'market_id': 340, 'market_name': 'Saharanpur'}, {'market_id': 1929, 'market_name': 'Gangoh'}]}


In [22]:
district_id = 132

for market in district_market_map["132"]["markets"]:

    market_id = market["market_id"]
    market_name = market["market_name"]

    payload = {
        "commodity_id": COMMODITY_ID,
        "state_id": STATE_ID,
        "district_id": district_id,
        "market_id": market_id,
        "from_date": "2024-01-01",
        "to_date": "2025-12-31"
    }

    resp = post_with_retry(
        "prices",
        payload
    )

    try:
        print(
            market_name,
            "->",
            resp["output"]["message"]
        )
    except:
        print(market_name, resp)

Nakud -> No data exists
Rampurmaniharan -> Data exists
Deoband -> Data exists
Chutmalpur -> No data exists
Sultanpurchilkana -> No data exists
Nanuta -> No data exists
Saharanpur -> Data exists
Rate limit on key 2

Switched to API key 3
Gangoh -> No data exists


In [23]:
payload = {
    "commodity_id": COMMODITY_ID,
    "state_id": STATE_ID,
    "district_id": 132,
    "market_id": 1960,   # Rampurmaniharan
    "from_date": "2024-01-01",
    "to_date": "2025-12-31"
}

resp = post_with_retry("prices", payload)

print(resp)

{'output': {'type': 'success', 'message': 'Data exists', 'data': [{'date': '2024-05-17T00:00:00.000Z', 'commodity_id': 1, 'census_state_id': 9, 'census_district_id': 132, 'market_id': 1960, 'min_price': 2290, 'max_price': 2330, 'modal_price': 2300}, {'date': '2024-06-28T00:00:00.000Z', 'commodity_id': 1, 'census_state_id': 9, 'census_district_id': 132, 'market_id': 1960, 'min_price': 2275, 'max_price': 2500, 'modal_price': 2400}, {'date': '2024-07-26T00:00:00.000Z', 'commodity_id': 1, 'census_state_id': 9, 'census_district_id': 132, 'market_id': 1960, 'min_price': 2300, 'max_price': 2550, 'modal_price': 2480}, {'date': '2025-01-31T00:00:00.000Z', 'commodity_id': 1, 'census_state_id': 9, 'census_district_id': 132, 'market_id': 1960, 'min_price': 2850, 'max_price': 3100, 'modal_price': 3000}, {'date': '2025-04-25T00:00:00.000Z', 'commodity_id': 1, 'census_state_id': 9, 'census_district_id': 132, 'market_id': 1960, 'min_price': 2400, 'max_price': 2500, 'modal_price': 2450}, {'date': '2025

In [ ]:
df = pd.DataFrame(resp["output"]["data"])

print(df.columns)
print(df.head())

Index(['date', 'commodity_id', 'census_state_id', 'census_district_id',
       'market_id', 'min_price', 'max_price', 'modal_price'],
      dtype='str')
                       date  commodity_id  census_state_id  \
0  2024-05-17T00:00:00.000Z             1                9   
1  2024-06-28T00:00:00.000Z             1                9   
2  2024-07-26T00:00:00.000Z             1                9   
3  2025-01-31T00:00:00.000Z             1                9   
4  2025-04-25T00:00:00.000Z             1                9   

   census_district_id  market_id  min_price  max_price  modal_price  
0                 132       1960       2290       2330         2300  
1                 132       1960       2275       2500         2400  
2                 132       1960       2300       2550         2480  
3                 132       1960       2850       3100         3000  
4                 132       1960       2400       2500         2450  


In [25]:
import json

with open("district_market_map.json") as f:
    district_market_map = json.load(f)

for district_id, info in district_market_map.items():

    print("\n" + "=" * 60)
    print(f"{info['name']} ({district_id})")
    print("=" * 60)

    for market in info["markets"]:
        print(
            f"{market['market_id']:>6}  "
            f"{market['market_name']}"
        )


Saharanpur (132)
  2544  Nakud
  1960  Rampurmaniharan
  1957  Deoband
  2543  Chutmalpur
  2546  Sultanpurchilkana
  2545  Nanuta
   340  Saharanpur
  1929  Gangoh

Muzaffarnagar (133)
   677  Khatauli
  2548  Thanabhavan
   676  Shamli
  2549  Shahpur
   709  Muzzafarnagar

Bijnor (134)
  1191  Bijnaur
  1916  Chaandpur
  1966  Najibabad
  2565  Dhampur
  2566  Kiratpur
  2567  Nagina
  2568  Haldaur

Moradabad (135)
   337  Muradabad
   682  Chandausi
  1961  Bhehjoi
  1963  Sambhal

Rampur (136)
   680  Rampur
  1908  Vilaspur
  2570  Tanda(Rampur)
  2571  Shahabad
  3452  Milak

Jyotiba Phule Nagar (137)
  1193  Amroha
  1962  Hasanpur
  2569  Dhanaura

Meerut (138)
   336  Meerut
  1959  Sardhana
  2532  Mawana
  2533  Parikshitgarh

Baghpat (139)
   675  Baraut
  2535  Khekda

Ghaziabad (140)
   324  Ghaziabad
  1900  Hapur
  3446  Noida
  2534  Muradnagar

Gautam Buddha Nagar (141)
  1903  Dadri
  2541  Dankaur
  2542  Javer

Bulandshahr (142)
  2536  Jahangirabad
  1958  Gula

In [4]:
# =====================================
# CONFIG
# =====================================

OUTDIR = Path("UP_WHEAT_2006_2025")
OUTDIR.mkdir(exist_ok=True)

with open("district_market_map.json") as f:
    district_market_map = json.load(f)

MASTER_FILE = (
    OUTDIR /
    "UP_WHEAT_2006_2025_MASTER.csv"
)

In [5]:
print(
    f"Loaded "
    f"{len(district_market_map)} districts"
)

Loaded 71 districts


In [6]:
print(len(district_market_map))
print(MASTER_FILE)

71
UP_WHEAT_1998_2025/UP_WHEAT_1998_2025_MASTER.csv


In [5]:
import json

KNOWN_EMPTY_PATH = OUTDIR / "_known_empty_markets.json"

def load_known_empty():
    if KNOWN_EMPTY_PATH.exists():
        with open(KNOWN_EMPTY_PATH) as f:
            return set(tuple(x) for x in json.load(f))
    return set()

def save_known_empty(known_empty):
    with open(KNOWN_EMPTY_PATH, "w") as f:
        json.dump(sorted(known_empty), f)

known_empty = load_known_empty()

In [6]:
print(
    f"Starting download with key "
    f"{current_key_idx + 1}"
)

print(
    f"Active API key: "
    f"{current_key_idx + 1}"
)

download_interrupted = False

total_districts = len(district_market_map)
district_counter = 0

# =====================================
# DOWNLOAD LOOP
# =====================================

for district_id, info in district_market_map.items():

    district_counter += 1

    district_id = int(district_id)

    district_name = info["name"]

    district_dir = (
        OUTDIR / district_name
    )

    district_dir.mkdir(
        exist_ok=True
    )

    print(
        "\n" + "=" * 60
    )

    print(
    f"[{district_counter}/{total_districts}] "
    f"{district_name}"
)

    print(
        "=" * 60
    )

    for year in range(
        2006,
        2026
    ):

        outfile = (
            district_dir /
            f"{year}.csv"
        )

        # =================================
        # SKIP IF ALREADY DOWNLOADED
        # =================================

        if outfile.exists():

            print(
                f"Skipping "
                f"{district_name} "
                f"{year}"
            )

            continue

        print(f"\nProcessing {district_name} {year}")

        all_empty = all(
            (district_id, m["market_id"], year) in known_empty
            for m in info["markets"]
        )

        if all_empty:
            print(f"Skipping {district_name} {year} (known empty)")
            continue

        district_year_frames = []

        # =================================
        # MARKET LOOP
        # =================================

        for market in info["markets"]:

            market_id = market[
                "market_id"
            ]

            market_name = market[
                "market_name"
            ]

            if (district_id, market_id, year) in known_empty:
                continue

            payload = {

                "commodity_id":
                    COMMODITY_ID,

                "state_id":
                    STATE_ID,

                "district_id":
                    district_id,

                "market_id":
                    market_id,

                "from_date":
                    f"{year}-01-01",

                "to_date":
                    f"{year}-12-31"
            }

            print(
                f"[KEY {current_key_idx+1}] "
                f"{district_name} | "
                f"{year} | "
                f"{market_name}"
            )

            resp = post_with_retry(
                "prices",
                payload
            )

            # =================================
            # REQUEST FAILED (NETWORK/SERVER)
            # =================================

            if (
                isinstance(resp, dict)
                and
                resp.get("status")
                == "request_failed"
            ):

                print(
                    f"Request failed: "
                    f"{district_name} | "
                    f"{market_name} | "
                    f"{year}"
                )

                continue

            # =================================
            # ALL KEYS EXHAUSTED
            # =================================

            if resp is None:

                print(
                    "\nStopping because "
                    "all API keys are exhausted."
                )

                print("\nKEY USAGE SUMMARY")

                for k, v in key_stats.items():

                    print(
                        f"Key {k+1}: "
                        f"{v['success_requests']} "
                        f"successful requests"
                    )

                download_interrupted = True

                break

            try:

                output = resp.get("output", {})

                if (
                    output.get("message")
                    == "No data exists"
                ):
                    known_empty.add((district_id, market_id, year))
                    save_known_empty(known_empty)
                    continue

                records = output.get("data", [])

                if len(records) == 0:
                    known_empty.add((district_id, market_id, year))
                    save_known_empty(known_empty)
                    continue

                df = pd.DataFrame(records)
                df["date"] = pd.to_datetime(df["date"])
                df["district_name"] = district_name
                df["market_name"] = market_name
                df["year"] = df["date"].dt.year

                district_year_frames.append(df)

                print(
                    f"[KEY {current_key_idx+1}] "
                    f"{market_name}: "
                    f"{len(df)} rows"
                )

            except Exception as e:
                print(district_name, market_name, year, e)
        
        if download_interrupted:
            break

        # =================================
        # SAVE DISTRICT-YEAR FILE
        # =================================

        if district_year_frames:

            year_df = pd.concat(
                district_year_frames,
                ignore_index=True
            )

            year_df = (
                year_df
                .sort_values(
                    [
                        "market_name",
                        "date"
                    ]
                )
            )

            year_df.to_csv(
                outfile,
                index=False
            )

            print(
                f"Saved "
                f"{outfile}"
            )

        else:

            print(
                f"No data found for "
                f"{district_name} "
                f"{year}"
            )
        
    if download_interrupted:
        break

# =====================================
# FINAL SUMMARY
# =====================================

print("\n")
print("=" * 60)
print("DOWNLOAD COMPLETED")
print("=" * 60)

print(
    f"CSV files created: "
    f"{len(list(OUTDIR.rglob('*.csv')))}"
)

total = 0

for idx, stats in key_stats.items():

    total += stats[
        "success_requests"
    ]

    print(
        f"Key {idx+1}: "
        f"{stats['success_requests']} "
        f"successful requests"
    )

print(
    f"\nTotal successful requests: "
    f"{total}"
)

print(
    f"\nOutput folder:"
)

print(OUTDIR)

Starting download with key 1
Active API key: 1

[1/71] Saharanpur
Skipping Saharanpur 2006
Skipping Saharanpur 2007
Skipping Saharanpur 2008
Skipping Saharanpur 2009
Skipping Saharanpur 2010
Skipping Saharanpur 2011
Skipping Saharanpur 2012
Skipping Saharanpur 2013
Skipping Saharanpur 2014
Skipping Saharanpur 2015
Skipping Saharanpur 2016
Skipping Saharanpur 2017
Skipping Saharanpur 2018
Skipping Saharanpur 2019
Skipping Saharanpur 2020
Skipping Saharanpur 2021
Skipping Saharanpur 2022
Skipping Saharanpur 2023
Skipping Saharanpur 2024
Skipping Saharanpur 2025

[2/71] Muzaffarnagar
Skipping Muzaffarnagar 2006
Skipping Muzaffarnagar 2007
Skipping Muzaffarnagar 2008
Skipping Muzaffarnagar 2009
Skipping Muzaffarnagar 2010
Skipping Muzaffarnagar 2011
Skipping Muzaffarnagar 2012
Skipping Muzaffarnagar 2013
Skipping Muzaffarnagar 2014
Skipping Muzaffarnagar 2015
Skipping Muzaffarnagar 2016
Skipping Muzaffarnagar 2017
Skipping Muzaffarnagar 2018
Skipping Muzaffarnagar 2019
Skipping Muzaffarnag